In [24]:
import anndata as ad
import scanpy as sc
import pandas as pd
from LingerGRN.pseudo_bulk import *
import LingerGRN.LINGER_tr as LINGER_tr
import LingerGRN.LL_net as LL_net
from LingerGRN.preprocess import *

# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Data read

In [17]:
cell_type = "HepG2"
data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/" + cell_type + "/process/"
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/" + cell_type + "/data_linger/"
os.makedirs(output_path, exist_ok=True)

adata_RNA = ad.read_h5ad(data_path + "rna_processed.h5ad")
adata_ATAC = ad.read_h5ad(data_path + "atac_processed.h5ad")

def fix_peak_format(name):
    if ':' not in name and '-' in name:
        return name.replace('-', ':', 1)
    return name

adata_ATAC.var_names = [fix_peak_format(name) for name in adata_ATAC.var_names]

# Data process

In [18]:

sc.pp.filter_cells(adata_RNA, min_genes=200)
sc.pp.filter_genes(adata_RNA, min_cells=3)
sc.pp.filter_cells(adata_ATAC, min_genes=200)
sc.pp.filter_genes(adata_ATAC, min_cells=3)

selected_cell=list(set(adata_RNA.obs.index)&set(adata_ATAC.obs.index))

adata_RNA = adata_RNA[selected_cell,]
adata_ATAC = adata_ATAC[selected_cell,]


adata_RNA.obs['barcode'] = adata_RNA.obs_names
adata_ATAC.obs['barcode'] = adata_ATAC.obs_names

adata_RNA.obs['sample'] = 'sample_1'
adata_ATAC.obs['sample'] = 'sample_1'
adata_RNA.obs['label'] = '0' 
adata_ATAC.obs['label'] = '0'
adata_ATAC.var['gene_ids'] = adata_ATAC.var_names
adata_RNA.var['gene_ids'] = adata_RNA.var_names

Trying to modify attribute `.obs` of view, initializing view as actual.
Trying to modify attribute `.obs` of view, initializing view as actual.


In [19]:

samplelist=list(set(adata_ATAC.obs['sample'].values)) # sample is generated from cell barcode 
tempsample=samplelist[0]

TG_pseudobulk=pd.DataFrame([])
RE_pseudobulk=pd.DataFrame([])
singlepseudobulk = (adata_RNA.obs['sample'].unique().shape[0]*adata_RNA.obs['sample'].unique().shape[0]>100)
for tempsample in samplelist:
    adata_RNAtemp=adata_RNA[adata_RNA.obs['sample']==tempsample]
    adata_ATACtemp=adata_ATAC[adata_ATAC.obs['sample']==tempsample]
    TG_pseudobulk_temp,RE_pseudobulk_temp=pseudo_bulk(adata_RNAtemp,adata_ATACtemp,singlepseudobulk)                
    TG_pseudobulk=pd.concat([TG_pseudobulk, TG_pseudobulk_temp], axis=1)
    RE_pseudobulk=pd.concat([RE_pseudobulk, RE_pseudobulk_temp], axis=1)
    RE_pseudobulk[RE_pseudobulk > 100] = 100

adata_ATAC.write(output_path + 'adata_ATAC.h5ad')
adata_RNA.write(output_path + 'adata_RNA.h5ad')
TG_pseudobulk=TG_pseudobulk.fillna(0)
RE_pseudobulk=RE_pseudobulk.fillna(0)
pd.DataFrame(adata_ATAC.var.index).to_csv(output_path + 'Peaks.txt',header=None,index=None)
TG_pseudobulk.to_csv(output_path + 'TG_pseudobulk.tsv')
RE_pseudobulk.to_csv(output_path + 'RE_pseudobulk.tsv')

Received a view of an AnnData. Making a copy.
Received a view of an AnnData. Making a copy.
Received a view of an AnnData. Making a copy.
Received a view of an AnnData. Making a copy.


# Train

In [20]:
Datadir=output_path
GRNdir='/home/liyang/BioWuYan/dygmamba_project/data/Data_LINGER/data_bulk/'
genome='hg38'
outdir = output_path + "train/"
os.makedirs(outdir, exist_ok=True)
method='LINGER'
preprocess(TG_pseudobulk,RE_pseudobulk,GRNdir,genome,method,output_path, outdir)


activef='ReLU' # active function chose from 'ReLU','sigmoid','tanh'
LINGER_tr.training(GRNdir,method,outdir,activef,'Human')

LL_net.TF_RE_binding(GRNdir,adata_RNA,adata_ATAC,genome,method,Datadir, outdir)


Mapping gene expression...
Generate TF expression...
Generate RE chromatin accessibility...
Generate TF binding...


100%|██████████| 23/23 [07:20<00:00, 19.17s/it]


Generate Index...


100%|██████████| 1818/1818 [00:00<00:00, 3568.38it/s]


chr1


...

100%|██████████| 47/47 [00:17<00:00,  2.63it/s]


Generating cellular population TF binding strength ...


  0%|          | 0/23 [00:00<?, ?it/s]


Generating cellular population TF binding strength for chr1


FileNotFoundError: [Errno 2] No such file or directory: '/home/liyang/BioWuYan/dygmamba_project/data/cell_line/HepG2/data_linger/train/Peaks.txt'

In [26]:
LL_net.cis_reg(GRNdir,adata_RNA,adata_ATAC,genome,method,output_path, outdir)


100%|██████████| 23/23 [00:01<00:00, 13.82it/s]


In [28]:
LL_net.trans_reg(GRNdir,method,output_path, outdir,genome)

Generate trans-regulatory netowrk ...


100%|██████████| 23/23 [00:01<00:00, 16.27it/s]


Save trans-regulatory netowrk ...
